# ODIR-5K Normal vs Abnormal: 5-fold cross-validation
### PyTorch · EfficientNet-B0 · 512px · v2 regularisation · patient-grouped folds

This notebook takes the **v2 setup** and measures it more reliably:
- **Same fixed test set** as v1 and v2: the same 20% of patients, held out with the same seed. The numbers are directly comparable.
- The other 80% of patients are split into **5 patient-grouped folds**. Each fold trains one model on 4 folds and validates on the 5th, which is used for early stopping and threshold tuning. Every model is then evaluated on the test set.
- **Reported results:**
  - per-fold test metrics as **mean ± standard deviation**;
  - a **5-model ensemble** that averages the five models' probabilities;
  - a patient-level **bootstrap 95% CI** for the ensemble's test AUC.

**Run on Kaggle**
1. Import this notebook.
2. *Session options*: **GPU T4 x2** (not P100) and **Internet on**. The dataset is fetched automatically, or attach `andrewmvd/ocular-disease-recognition-odir5k` with *Add Input*.
3. Use **Save Version → Save & Run All (Commit)**, so it keeps running if you close the tab. It takes about **3–3.5 hours**. Then open the version's log or output to get the notebook and files.

**Run on your own GPU**
- Install PyTorch with CUDA plus `pandas scikit-learn matplotlib seaborn tqdm pillow kagglehub jupyter ipywidgets`.
- Results go to `outputs_cv/` next to the notebook.
- With 8 GB of GPU memory, set `BATCH_SIZE = 16`.
- On Windows, `NUM_WORKERS` is set to 0 automatically, which is slower. WSL2 or Linux is faster.

**Resumable.** Each finished fold is saved. If the run stops, run all cells again and finished folds are skipped. *(On Kaggle this only works within the same session or commit.)*

Baselines on this test set: **v1** (384px) ROC-AUC 0.846 · **v2** (512px, stronger regularisation) ROC-AUC 0.861.

## 1. Setup

In [ ]:
import os, re, gc, random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, roc_curve, average_precision_score, f1_score,
                             accuracy_score, confusion_matrix)

warnings.filterwarnings('ignore')

In [ ]:
# ----------------------------- Config -----------------------------
DATASET  = 'andrewmvd/ocular-disease-recognition-odir5k'   # kaggle.com/datasets/andrewmvd/ocular-disease-recognition-odir5k
DATA_DIR = Path('/kaggle/input/ocular-disease-recognition-odir5k')   # or the folder containing full_df.csv on your PC
OUT_DIR  = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs_cv')

IMG_SIZE    = 512
BATCH_SIZE  = 32        # use 16 (or 24) if you get CUDA out-of-memory
NUM_WORKERS = 0 if os.name == 'nt' else min(4, os.cpu_count() or 2)   # Windows notebooks need 0
SEED        = 42
N_FOLDS     = 5

FROZEN_EPOCHS   = 3
FINETUNE_EPOCHS = 30
PATIENCE        = 7
LR_HEAD         = 1e-3
LR_BACKBONE     = 1e-4
WEIGHT_DECAY    = 0.05
DROPOUT         = 0.5
MIXUP_ALPHA     = 0.4
MIXUP_PROB      = 0.5
TARGET_SENSITIVITY = 0.85

CLASS_NAMES = ['Normal', 'Abnormal']   # target 0 = Normal, 1 = Abnormal

In [ ]:
def find_dataset():
    """Attached dataset (any folder under /kaggle/input), a local DATA_DIR, or download with kagglehub."""
    if (DATA_DIR / 'full_df.csv').exists():
        return DATA_DIR
    root = Path('/kaggle/input')
    hits = sorted(root.rglob('full_df.csv')) if root.exists() else []
    if hits:
        return hits[0].parent
    print(f'Dataset not found locally - fetching {DATASET} with kagglehub (may take a few minutes) ...')
    try:
        import kagglehub
        path = Path(kagglehub.dataset_download(DATASET))
    except Exception as e:
        raise RuntimeError(f'Could not fetch {DATASET}. On Kaggle turn on Internet; on your PC set up a Kaggle '
                           'API token (kaggle.json) or download the dataset and set DATA_DIR.') from e
    hits = sorted(path.rglob('full_df.csv'))
    if not hits:
        raise FileNotFoundError(f'full_df.csv not found in {path}')
    return hits[0].parent

DATA_DIR = find_dataset()
CSV_PATH, IMG_DIR = DATA_DIR / 'full_df.csv', DATA_DIR / 'preprocessed_images'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(SEED)
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
torch.backends.cudnn.benchmark = True
if USE_AMP:   # fail early if this PyTorch build can't run on the GPU (e.g. Kaggle P100)
    try:
        torch.ones(1, device=device).add_(1); torch.cuda.synchronize()
    except RuntimeError as e:
        raise RuntimeError(f'PyTorch {torch.__version__} cannot run on {torch.cuda.get_device_name(0)}. '
                           'On Kaggle choose "GPU T4 x2".') from e

print('torch', torch.__version__, '| device:', device,
      f'({torch.cuda.get_device_name(0)})' if USE_AMP else '- NO GPU: this would take days on CPU')
print('data :', DATA_DIR); print('out  :', OUT_DIR.resolve()); print('workers:', NUM_WORKERS)

## 2. Per-eye labels (same as v1/v2)

Each image is labelled from its own eye's keywords:
- **Normal:** the only diagnosis is `normal fundus`. Artifacts such as `lens dust` are ignored.
- **Abnormal:** at least one disease keyword.
- **Dropped:** unusable images (`low image quality`, `no fundus image`, `anterior segment image`).

In [ ]:
raw = pd.read_csv(CSV_PATH)
patients = raw.drop_duplicates(subset='ID').reset_index(drop=True)

UNUSABLE  = {'low image quality', 'no fundus image', 'anterior segment image'}
ARTIFACTS = {'lens dust', 'image offset', 'optic disk photographically invisible'}

def split_keywords(text):
    if pd.isna(text):
        return []
    return [t.strip() for t in re.split(r'[，,、;；]', str(text).lower()) if t.strip()]

def eye_label(keywords):
    terms = set(split_keywords(keywords))
    if terms & UNUSABLE:
        return None
    diagnoses = terms - ARTIFACTS
    if not diagnoses:
        return None
    return 0 if diagnoses == {'normal fundus'} else 1

rows = []
for _, r in patients.iterrows():
    for side in ('Left', 'Right'):
        path = IMG_DIR / r[f'{side}-Fundus']
        y = eye_label(r[f'{side}-Diagnostic Keywords'])
        if path.exists() and y is not None:
            rows.append({'ID': r['ID'], 'eye': side.lower(), 'file_path': str(path),
                         'keywords': ', '.join(split_keywords(r[f'{side}-Diagnostic Keywords'])), 'target': y})
eyes = pd.DataFrame(rows)
print(f'{len(eyes)} labelled eye images from {eyes.ID.nunique()} patients | abnormal = {eyes.target.mean():.1%}')

## 3. Fixed test set + 5 patient-grouped folds

The test split is the same call with the same seed as v1/v2, so it holds the **same patients**. The remaining patients are divided into `N_FOLDS` folds. Each patient's images, and both of their eyes, stay in a single fold.

In [ ]:
def group_split(df, n_splits, seed):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    keep_idx, hold_idx = next(sgkf.split(df, df['target'], groups=df['ID']))
    return df.iloc[keep_idx].reset_index(drop=True), df.iloc[hold_idx].reset_index(drop=True)

trainval_df, test_df = group_split(eyes, 5, SEED)          # identical to v1/v2

trainval_df['fold'] = -1
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (_, va_idx) in enumerate(sgkf.split(trainval_df, trainval_df['target'], groups=trainval_df['ID'])):
    trainval_df.loc[va_idx, 'fold'] = k

assert (trainval_df.fold >= 0).all()
assert trainval_df.groupby('ID').fold.nunique().max() == 1, 'a patient is in two folds'
assert not set(trainval_df.ID) & set(test_df.ID), 'patient leakage into test'

summary = trainval_df.groupby('fold').agg(images=('target', 'size'), patients=('ID', 'nunique'),
                                          abnormal=('target', 'mean'))
summary.loc['test'] = [len(test_df), test_df.ID.nunique(), test_df.target.mean()]
summary['abnormal'] = (summary['abnormal'] * 100).round(1).astype(str) + '%'
display(summary)

## 4. Data pipeline, model and training functions (same settings as v2)

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tfms = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(30),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.03),
    T.ToTensor(), T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.08)),
])
eval_tfms = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

class FundusDataset(Dataset):
    def __init__(self, df, transform):
        self.paths, self.targets, self.transform = df['file_path'].tolist(), df['target'].astype(int).tolist(), transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.transform(Image.open(self.paths[i]).convert('RGB')), self.targets[i]

def make_loader(df, tfms, shuffle):
    return DataLoader(FundusDataset(df, tfms), batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=USE_AMP, drop_last=shuffle, persistent_workers=NUM_WORKERS > 0)

def build_model(pretrained=True):
    try:
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
    except Exception as e:
        raise RuntimeError('Could not download the ImageNet weights - turn on Internet and re-run.') from e
    model.classifier = nn.Sequential(nn.Dropout(p=DROPOUT), nn.Linear(model.classifier[1].in_features, 2))
    return model.to(device)

def set_backbone_trainable(model, trainable):
    for p in model.features.parameters():
        p.requires_grad = trainable

def run_epoch(model, loader, criterion, scaler, optimizer=None):
    training = optimizer is not None
    model.train(training)
    if training and not next(model.features.parameters()).requires_grad:
        model.features.eval()                         # frozen backbone: keep BatchNorm statistics fixed
    total_loss, probs, targets = 0.0, [], []
    for x, y in tqdm(loader, leave=False, desc='train' if training else 'eval'):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        lam, y_b = 1.0, y
        if training and MIXUP_ALPHA > 0 and np.random.rand() < MIXUP_PROB:
            lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)); lam = max(lam, 1 - lam)
            perm = torch.randperm(x.size(0), device=x.device)
            x, y_b = lam * x + (1 - lam) * x[perm], y[perm]
        with torch.set_grad_enabled(training), torch.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(x)
            loss = lam * criterion(logits, y) + (1 - lam) * criterion(logits, y_b)
        if training:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * x.size(0)
        probs.append(torch.softmax(logits.float(), 1)[:, 1].detach().cpu()); targets.append(y.cpu())
    probs, targets = torch.cat(probs).numpy(), torch.cat(targets).numpy()
    return total_loss / len(probs), roc_auc_score(targets, probs)

@torch.no_grad()
def predict_probs(model, loader):
    """P(abnormal) with horizontal-flip test-time augmentation."""
    model.eval(); out = []
    for x, _ in tqdm(loader, leave=False, desc='predict'):
        x = x.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            p = (torch.softmax(model(x).float(), 1)[:, 1] + torch.softmax(model(torch.flip(x, dims=[3])).float(), 1)[:, 1]) / 2
        out.append(p.float().cpu())
    return torch.cat(out).numpy()

test_loader = make_loader(test_df, eval_tfms, shuffle=False)

In [ ]:
def train_fold(k):
    """Train the model for fold k; save its best weights, history, validation and test predictions."""
    ckpt, val_file, test_file = OUT_DIR / f'fold{k}_best.pt', OUT_DIR / f'fold{k}_val_preds.csv', OUT_DIR / f'fold{k}_test_preds.csv'
    if ckpt.exists() and val_file.exists() and test_file.exists():
        print(f'fold {k}: already done - skipping'); return

    seed_everything(SEED + k)
    tr_df, va_df = trainval_df[trainval_df.fold != k], trainval_df[trainval_df.fold == k]
    train_loader, val_loader = make_loader(tr_df, train_tfms, True), make_loader(va_df, eval_tfms, False)
    model = build_model()
    counts = np.bincount(tr_df['target'], minlength=2)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32, device=device),
                                    label_smoothing=0.05)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    state, history = dict(auc=-1.0, loss=float('inf'), epoch=0, bad=0), []
    print(f'===== fold {k}: {len(tr_df)} train / {len(va_df)} val images =====')

    def fit(optimizer, scheduler, n_epochs, stage):
        for _ in range(n_epochs):
            t0 = time.time()
            tr_loss, tr_auc = run_epoch(model, train_loader, criterion, scaler, optimizer)
            va_loss, va_auc = run_epoch(model, val_loader, criterion, scaler)
            if scheduler is not None:
                scheduler.step()
            epoch = len(history) + 1
            history.append(dict(epoch=epoch, stage=stage, train_loss=tr_loss, val_loss=va_loss, train_auc=tr_auc, val_auc=va_auc))
            improved = va_auc > state['auc'] or (va_auc == state['auc'] and va_loss < state['loss'])
            if improved:
                state.update(auc=va_auc, loss=va_loss, epoch=epoch, bad=0); torch.save(model.state_dict(), ckpt)
            else:
                state['bad'] += 1
            print(f'[fold {k} {stage:8s}] epoch {epoch:2d} | train loss {tr_loss:.4f} auc {tr_auc:.4f} | '
                  f'val loss {va_loss:.4f} auc {va_auc:.4f} | {time.time() - t0:4.0f}s' + ('  <- best' if improved else ''))
            if stage == 'finetune' and state['bad'] >= PATIENCE:
                print(f'fold {k}: early stopping'); break

    set_backbone_trainable(model, False)
    fit(torch.optim.AdamW(model.classifier.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY), None, FROZEN_EPOCHS, 'frozen')
    set_backbone_trainable(model, True)
    opt = torch.optim.AdamW([{'params': model.features.parameters(), 'lr': LR_BACKBONE},
                             {'params': model.classifier.parameters(), 'lr': LR_HEAD}], weight_decay=WEIGHT_DECAY)
    state['bad'] = 0
    fit(opt, torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FINETUNE_EPOCHS, eta_min=1e-6), FINETUNE_EPOCHS, 'finetune')
    print(f'fold {k}: best val AUC {state["auc"]:.4f} at epoch {state["epoch"]}')

    model.load_state_dict(torch.load(ckpt, map_location=device))
    pd.DataFrame(history).to_csv(OUT_DIR / f'fold{k}_history.csv', index=False)
    va_out = va_df[['ID', 'eye', 'file_path', 'target']].copy(); va_out['prob'] = predict_probs(model, val_loader)
    te_out = test_df[['ID', 'eye', 'file_path', 'target']].copy(); te_out['prob'] = predict_probs(model, test_loader)
    va_out.to_csv(val_file, index=False); te_out.to_csv(test_file, index=False)   # written last = fold complete

    del model, train_loader, val_loader; gc.collect()
    if USE_AMP: torch.cuda.empty_cache()

## 5. Train all folds (about 35–45 minutes per fold on a T4)

In [ ]:
t_start = time.time()
for k in range(N_FOLDS):
    train_fold(k)
print(f'All folds done in {(time.time() - t_start) / 3600:.2f} h')

## 6. Results

- **Per fold:** each model is scored on the test set at three thresholds: 0.5, Youden, and 85% sensitivity. Thresholds are tuned on that fold's own validation predictions.
- **Ensemble:** the five models' test probabilities are averaged. Its thresholds are tuned on the pooled out-of-fold validation predictions, so the test set is never used for tuning.

In [ ]:
def thresholds_from(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    youden = float(np.clip(thr[np.argmax(tpr - fpr)], 0, 1))
    sens   = float(np.clip(thr[np.argmax(tpr >= TARGET_SENSITIVITY)], 0, 1))
    return {'0.5': 0.5, 'Youden': youden, f'{TARGET_SENSITIVITY:.0%} sens': sens}

def metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {'roc_auc': roc_auc_score(y, p), 'pr_auc': average_precision_score(y, p),
            'accuracy': accuracy_score(y, pred), 'f1_abnormal': f1_score(y, pred, zero_division=0),
            'sensitivity': tp / max(tp + fn, 1), 'specificity': tn / max(tn + fp, 1), 'threshold': thr}

val_preds  = [pd.read_csv(OUT_DIR / f'fold{k}_val_preds.csv') for k in range(N_FOLDS)]
test_preds = [pd.read_csv(OUT_DIR / f'fold{k}_test_preds.csv') for k in range(N_FOLDS)]
y_test = test_df['target'].values
for tp_ in test_preds:
    assert (tp_['file_path'].values == test_df['file_path'].values).all()

rows = []
for k in range(N_FOLDS):
    thr = thresholds_from(val_preds[k]['target'].values, val_preds[k]['prob'].values)
    for name, t in thr.items():
        rows.append({'fold': k, 'rule': name, 'val_auc': roc_auc_score(val_preds[k]['target'], val_preds[k]['prob']),
                     **metrics(y_test, test_preds[k]['prob'].values, t)})
per_fold = pd.DataFrame(rows)
per_fold.to_csv(OUT_DIR / 'cv_per_fold_test_metrics.csv', index=False)

cols = ['val_auc', 'roc_auc', 'pr_auc', 'accuracy', 'f1_abnormal', 'sensitivity', 'specificity']
mean_sd = per_fold.groupby('rule', sort=False)[cols].agg(lambda s: f'{s.mean():.4f} ± {s.std(ddof=1):.4f}')
print('Per-fold models on the TEST set (mean ± sd over folds):')
display(mean_sd)

In [ ]:
# 5-model ensemble: average probabilities; thresholds from pooled out-of-fold validation predictions
oof = pd.concat(val_preds, ignore_index=True)
oof_auc = roc_auc_score(oof['target'], oof['prob'])
ens_prob = np.mean([tp_['prob'].values for tp_ in test_preds], axis=0)
ens_thr = thresholds_from(oof['target'].values, oof['prob'].values)

def bootstrap_auc_ci(y, p, groups, n_boot=1000, seed=SEED):
    """95% CI for ROC-AUC, resampling patients (both eyes together)."""
    rng = np.random.default_rng(seed)
    uniq = np.unique(groups); idx_of = {g: np.flatnonzero(groups == g) for g in uniq}
    aucs = []
    for _ in range(n_boot):
        idx = np.concatenate([idx_of[g] for g in rng.choice(uniq, size=len(uniq), replace=True)])
        if len(np.unique(y[idx])) == 2:
            aucs.append(roc_auc_score(y[idx], p[idx]))
    return np.percentile(aucs, [2.5, 97.5])

lo, hi = bootstrap_auc_ci(y_test, ens_prob, test_df['ID'].values)
ensemble = pd.DataFrame({name: metrics(y_test, ens_prob, t) for name, t in ens_thr.items()}).T.round(4)
ensemble.index = [f'ensemble @ {n}' for n in ensemble.index]
ensemble.to_csv(OUT_DIR / 'cv_ensemble_test_metrics.csv')
print(f'Out-of-fold validation AUC: {oof_auc:.4f}')
print(f'Ensemble TEST ROC-AUC: {roc_auc_score(y_test, ens_prob):.4f}  (95% CI {lo:.3f}-{hi:.3f}, patient bootstrap)')
display(ensemble)

ens_out = test_df[['ID', 'eye', 'file_path', 'keywords', 'target']].copy()
ens_out['prob_abnormal'] = ens_prob
for k in range(N_FOLDS):
    ens_out[f'prob_fold{k}'] = test_preds[k]['prob'].values
ens_out.to_csv(OUT_DIR / 'cv_ensemble_test_predictions.csv', index=False)

comparison = pd.DataFrame({
    'test ROC-AUC': ['0.846', '0.861',
                     f"{per_fold[per_fold.rule == '0.5'].roc_auc.mean():.3f} ± {per_fold[per_fold.rule == '0.5'].roc_auc.std(ddof=1):.3f}",
                     f'{roc_auc_score(y_test, ens_prob):.3f} ({lo:.3f}-{hi:.3f})']},
    index=['v1: 384px, single split', 'v2: 512px + regularisation, single split',
           f'CV: v2 setup, {N_FOLDS} models (mean ± sd)', f'CV: {N_FOLDS}-model ensemble (95% CI)'])
display(comparison)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(19, 5))
for k in range(N_FOLDS):
    h = pd.read_csv(OUT_DIR / f'fold{k}_history.csv')
    ax[0].plot(h.epoch, h.val_auc, label=f'fold {k}')
ax[0].axvline(FROZEN_EPOCHS + 0.5, ls='--', c='gray'); ax[0].set_title('Validation AUC per fold'); ax[0].set_xlabel('epoch'); ax[0].legend()

for k in range(N_FOLDS):
    f, t, _ = roc_curve(y_test, test_preds[k]['prob']); ax[1].plot(f, t, lw=1, alpha=0.6, label=f'fold {k} ({roc_auc_score(y_test, test_preds[k]["prob"]):.3f})')
f, t, _ = roc_curve(y_test, ens_prob); ax[1].plot(f, t, lw=2.5, c='black', label=f'ensemble ({roc_auc_score(y_test, ens_prob):.3f})')
ax[1].plot([0, 1], [0, 1], '--', c='gray'); ax[1].set_title('Test ROC curves')
ax[1].set_xlabel('1 - specificity'); ax[1].set_ylabel('sensitivity'); ax[1].legend(loc='lower right', fontsize=8)

key = f'{TARGET_SENSITIVITY:.0%} sens'
cm = confusion_matrix(y_test, (ens_prob >= ens_thr[key]).astype(int), labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax[2])
ax[2].set_xlabel('Predicted'); ax[2].set_ylabel('True'); ax[2].set_title(f'Ensemble @ {key} threshold ({ens_thr[key]:.2f})')
plt.tight_layout(); plt.savefig(OUT_DIR / 'cv_results.png', dpi=120); plt.show()

## 7. Using the ensemble on new images

In [ ]:
ens_models = []
for k in range(N_FOLDS):
    m = build_model(pretrained=False); m.load_state_dict(torch.load(OUT_DIR / f'fold{k}_best.pt', map_location=device))
    ens_models.append(m.eval())

torch.save({'state_dicts': [m.state_dict() for m in ens_models], 'thresholds': ens_thr, 'img_size': IMG_SIZE,
            'class_names': CLASS_NAMES, 'arch': 'efficientnet_b0', 'dropout': DROPOUT},
           OUT_DIR / 'odir_effnetb0_cv_ensemble.pt')

def predict(image_path, rule='Youden'):
    """P(abnormal) averaged over the fold models (with flip TTA); label from the chosen threshold rule."""
    x = eval_tfms(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        p = float(np.mean([(torch.softmax(m(x), 1)[0, 1] + torch.softmax(m(torch.flip(x, dims=[3])), 1)[0, 1]).item() / 2
                           for m in ens_models]))
    return {'file': Path(image_path).name, 'prob_abnormal': round(p, 4), 'rule': rule,
            'prediction': CLASS_NAMES[int(p >= ens_thr[rule])]}

for _, r in test_df.sample(4, random_state=0).iterrows():
    print(predict(r['file_path']), '| true:', CLASS_NAMES[r['target']])
print('Saved files:', sorted(p.name for p in OUT_DIR.iterdir() if p.suffix in {'.pt', '.csv', '.png'}))

## 8. What to report

- **Main result:** the **ensemble test ROC-AUC with its 95% CI**, plus sensitivity and specificity at the 85%-sensitivity threshold. That threshold was chosen on out-of-fold validation data, never on test.
- **Stability:** the **per-fold mean ± sd**. It shows how much a single model's score depends on which patients it was trained on.
- **Comparison with v1/v2:** all three use the same test patients. If the v1 or v2 AUC falls inside the ensemble's 95% CI, the difference isn't statistically clear.